In [1]:
import torch
# torch.autograd.set_detect_anomaly(True)
from torch.utils.data import DataLoader,WeightedRandomSampler
from torch.optim.lr_scheduler import PolynomialLR
import torch.nn.functional as F
from torchvision.transforms import v2
import pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2
import re
import segmentation_models_pytorch as smp
from pathlib import Path
from tqdm.notebook import tqdm
import json
import cv2
import matplotlib.pyplot as plt
import numpy as np
import random
import os
import gc
import zarr
import matplotlib
from collections import Counter
%load_ext autoreload
%autoreload 2
###IE###
from utils.helpers import (
    read_images,load_model,bootstrap_model,generate_all_masks,
    make_result_dataloaders,make_training_dataloaders,
    draw_fast_plots,count_class_pixels,effective_num_weights,
    left_mapping_function,TP_TN_FP_FN,get_collapse
)
from utils.preprocessing import (
    normalize_xca,normalize_empty,get_preprocessing,
    remove_black_border_multithread,BrightnessMultiplicativeNNUNet2D,
    ContrastAugmentationNNUNet2D
)
# from utils.debuging import show_bootstraping_results
from models.nnunet import nnUnet
from models.memory_unet import MemoryUnet
from models.swin_encoder import SwinUperNet,SwinUperNet2
from models.swin_upernet_cascade import SwinUperNetCascade
from models.hrnet import HRNET
from models.moe_model import MoeUnet
from models.S3_DSCNet_pro import DSCNet_pro
from models.smp_models import UnetPlPl,DeepLabV3Plus
from models.sarc_unet import SARC_UNet
from utils.testmodel import UTANet
from utils.losses import UnetLoss
from utils.recorder import HistoryRecorder
from scripts.logger import save_full_report
from scripts.trainer import trainer , evaluation
from scripts.image_clusters import valid,problem_maps
from utils.graph_extraction import (
    generate_robust_mask,
    create_gnn_dataset_parallel_limited
)
# from utils.option2_collaps import calc_combined_dice
from models.Swin_Unet.networks.vision_transformer import  SwinUnet
from utils.debuging import weighting_logic , save_loss_weights , save_weight_overlay_cv2
###SS###

/home/parsa/Venvs/AI/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
args = {
    "paths":{
        "base_path" : "./dataset/syntax/",
        "root_path" : "./dataset/syntax/topology_dataset/labels/",
        "output_base_path" : "./outputs",
        "images_folder" : "origianl_preprocessed", # new_preprocessed , origianl_preprocessed , images
        "clahe_folder" : "origianl_preprocessed",
        "generated_folder" : "generated_mask_probs",
        "labels_folder" : "labels",
        "ignore_labels_folder" : "k_fold_preds/2026-08-08_14-07-22/extra_labels",
        "bootstrap_folder" : "bootstrap",
        "hard_vessel_folder" : "hard_vessels_mask",
        "filter_folder" : "filters",
        "bootstrap_seq":"./bootstrap_sequence",
        "topology_folder" : "topology_dataset/labels",
        "side_folder":"./data/",
        "border_cords_folder" : "boarder_removed",
        "skel_folder" : "recall_skel"
        
    },
    "memory_unet":{
        "layer_count":4,
        "channel_max":256,
        "downsampling":"maxpool",
        "dropout" : False,
        "normaliztion":"InstanceNorm2d",#InstanceNorm2d
        "activation" : "LeakyReLU",
        "first_layer_out_c" : 32,
    },
    "swin":{
        "encoder_out_channels":150,
        "warmup_ep" : 0
    },
    "moe_unet" : {                 # 256,   128,    64,    32,     16,     8,      4
        # "encoder_out_channels" : [[128,2],[256,3],[512,5],[1024,2],[1024,3]],
        "encoder_out_channels":[[128,2],[256,3],[512,6],[1024,3],[1024,4]],
        "cbam_args":{"use_cbam":True,"ratio":16,"spatial_kernel":7},
        "dropout_args":{"use":True,"p":0.2},
        "resample_base_size" : None,
        "rp_out_channels" : 64,
        "router_counts":5,
        "expert_counts":3,
        "top_k":2,
        "train_moe":False,
        "norm" : "InstanceNorm2d"#InstanceNorm2d
    },
    "dsc_net":{
        "kernel_size" : 9,
        "extend_scope": 1,
        "if_offset" : True,
        "dim" : 1,
        "numer" : 16
    },
    "socre_params":{
        "p":0.4,
        "r":0.6,
        "choice_score" : "dice"
    },
    "image_shape" : (512,512),
    "in_c":3,
    "deep_super_vision":False,
    "bootstrap":False,
    "training_mode":"multi", # binary - multi - cleaning
    "binary_training_mode" : "partial",# full - partial
    "side_mode":"left", #both - left - right
    "report_class_wise":True,
    "attention" : False,
    "batch_size" : 4,
    "num_workers" : 4,
    "device" : "cuda" if torch.cuda.is_available() else "cpu",
    "epochs":50,
    "f_int_scale" : 2,
    "new_vessel_width":2,
    "full_report_cycle" : 5,
    "max_channels":512,
    "loss_type":"g_dice",
    "name" : "model[{}]-{}-{}-{}class-{}epo-Fold{}",
    "mask_ce_weights":None,
    "layer_count":5,
    "cut_borders":True,
    "use_ignore_mask" : True,
    ### Loss Functions
    "mask_ce_conf":{
        "imp_coef":1.0
    },
    "mask_frangi_ce_conf": {
        "expert_vessel_pos_weight": 4.0,
        "model_vessel_pos_weight": 3.0,
        "outer_boundary_boost": 4.5,
        "unlabeled_vessel_weight": 0.06,
        "bg_weight": 0.2,
        "kernel_size": 3,
        "imp_coef": 1.0,
        "use_expert_boundary_only": True
    },
    "mask_frangi_dice_conf": {
        "expert_vessel_pos_weight": 4.0,
        "model_vessel_pos_weight": 3.0,
        "outer_boundary_boost": 4.5,
        "unlabeled_vessel_weight": 0.06,
        "kernel_size": 3,
        "bg_weight": 0.3,
        "imp_coef": 1.0,
        "alpha": 0.2,
        "beta": 0.8,
        "use_expert_boundary_only": True
    },
    "ce_loss_conf":{
        "pos_weight":5.0,
        "bg_weight":1.0,
        "small_vessel_weights":10.0,
        "imp_coef":1.0
    },
    "fce_conf":{
        "gamma":2.0,
        "extra_w":3.0,
        "alpha":None
    },
    "tversky_conf":{
        "alpha":0.6,
        "beta":0.4,
        "gamma":1.00,
        "weights":None,
        "imp_coef":1.0
    },
    "g_dice_conf":{
        "imp_coef":1.0
    },
    "aux_loss_conf":{
        "imp_coef" : 0.1
    },
    "cl_dice_conf":{
        "imp_coef":1.0,
        "k":12
    },
    "recall_skel_coef":{
        "imp_coef":1.0
    },
    ### Optimizers
    "sgd_conf":{
        "lr" : 0.1,
        "momentum" : 0.99,
        "weight_decay" : 3e-5,
        "nesterov":True
    },
    "adam_conf":{
        "lr" : 0.01,
        "weight_decay" : 3e-5,
    },
    "adamw_conf":{
        "lr" : 0.0001,
        "weight_decay" : 0.01,
    }
}


if(args["training_mode"] == "binary" or args["training_mode"]=="cleaning"):
    args["class_count"] = 2 
    class_map = {
        1:"fg"
    }
    lut = None
    old_class_map = None,
    collapse = None
else:
    """
        sides : 
            0 - right side
            1 - left side
    """
    if(args["side_mode"] == "left"):
        
        old_class_map = {
            1: '5',2: '6',3: '7',4: '8',
            5: '9',6: '9a',7: '10',8: '10a',
            9: '11',10: '12',11: '12a',12: '13',
            13: '14',14: '14a',15: '15',
            16: '12b',17: '14b'
        }
        collapse , class_map =  get_collapse(option=0)
        lut = left_mapping_function(
            old_class_map = old_class_map,
            new_class_map = class_map,
            collapse = collapse
        )
        args["class_count"] = len(class_map) + 1
        
    elif(args["side_mode"] == "right"):
        # args["class_count"] = 9
        # class_map = {
        #     1: '1',2: '2', 3: '3',4: '4',
        #     5:'16',6: '16a',7: '16b',8: '16c'
        # }
        args["class_count"] = 6
        class_map = {
            1: '1',2: '2', 3: '3',4: '4',
            5:'16'
        }
        lut = None
    else:
        args["class_count"] = 26
        class_map = {
            1: '1',2: '2', 3: '3',4: '4',
            5: '5',6: '6',7: '7',8: '8',
            9: '9',10: '9a',11: '10',12: '10a',
            13: '11',14: '12',15: '12a',16: '13',
            17: '14',18: '14a',19: '15',20: '16',
            21: '16a',22: '16b',23: '16c',
            24: '12b',25: '14b'
        }
        lut = None


print("class_count : ",args["class_count"])


left_hard_classes = [10,11,14,15,17,18,19,24]#9a,10,12,12a,14,14a,15,12b
hard_classes = [20,21,22,23] + left_hard_classes
# losses_keys = ["total loss","FCE loss",args["loss_type"]]
losses_keys = ["total loss","mask CE-Loss",args["loss_type"],"recall skel"]#,"cl_dice_loss","mask CE-Loss" ,"branch_mask CE-Loss" ,args["loss_type"] ,"recall skel"
if(args["deep_super_vision"]):
    shape = args["image_shape"]
    # loss_weights = [1/(2**i) for i in range(1,len(args["moe_unet"]["encoder_out_channels"]))]
    loss_weights = [1/(2**i) for i in range(1,3)]
else:
    loss_weights = []
loss_weights



class_count :  4


[]

In [ ]:
full_train_images,sampler_weights,last_boot_level =  read_images(
    paths = args["paths"],
    part = "train",
    chosen_labels=None,
    hard_classes = hard_classes,
    bootstrap = args["bootstrap"],
    training_mode = args["training_mode"],
    binary_mode = args["binary_training_mode"],
    side_mode = args["side_mode"],
    in_c = args["in_c"],
    cut_borders = args["cut_borders"],
    class_count = args["class_count"],
    # lut = lut,
    collapse=collapse,
    old_class_map=old_class_map

)
full_valid_images = read_images(
    paths = args["paths"],
    part = "val",
    chosen_labels=None,
    bootstrap = args["bootstrap"],
    training_mode = args["training_mode"],
    binary_mode = args["binary_training_mode"],
    side_mode = args["side_mode"],
    in_c = args["in_c"],
    cut_borders = args["cut_borders"],
    class_count = args["class_count"],
    # lut = lut,
    collapse=collapse,
    old_class_map=old_class_map
)
test_images = read_images(
    paths = args["paths"],
    part = "test",
    chosen_labels=None,
    bootstrap = args["bootstrap"],
    training_mode = args["training_mode"],
    binary_mode = args["binary_training_mode"],
    side_mode = args["side_mode"],
    in_c = args["in_c"],
    cut_borders = args["cut_borders"],
    class_count = args["class_count"],
    # lut = lut,
    collapse=collapse,
    old_class_map=old_class_map
)

0


  0%|          | 0/1000 [00:00<?, ?it/s]

0


  0%|          | 0/200 [00:00<?, ?it/s]

0


  0%|          | 0/300 [00:00<?, ?it/s]

In [ ]:

train_co = {i+1:0 for i in range(len(class_map))}
valid_co = {}

In [4]:
def make_folds(names,K):
    val_size = int(len(names)/K)
    old_val_names = []
    folds = {}
    for i in range(K):
        val_names = names[i*val_size:(i+1)*val_size]
        train_names = names[(i+1)*val_size:] + old_val_names
        old_val_names+= val_names
        folds[i]= {"train":train_names,"val":val_names}
    return folds

def extract_fold_data(train_names,full_train_images):
    train_images = []
    valid_images = []
    for data in full_train_images:
        name = data["name_stem"]
        if(name in train_names):
            train_images.append(data)
        else:
            valid_images.append(data)
            
    return train_images,valid_images

In [5]:
train_transforms , test_transforms = get_preprocessing(args["training_mode"],args)

if(args["in_c"]==1):
    if(args["training_mode"]!="cleaning"):
        normalizer = normalize_xca
        print("set to z score")
    else:
        normalizer = normalize_empty
elif(args["in_c"]==2):
    print("set to z score 2c")
    normalizer = normalize_empty
else:
    normalizer = A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
        max_pixel_value=255.0,
    )
    print("set to imagenet")


set to imagenet


In [6]:
K=5
train_names = [data["name_stem"] for data in full_train_images]
random.shuffle(train_names)
folds = make_folds(train_names,K=K)

In [7]:
print("fold info : ")
for fold , data in folds.items():
    print(f"fold : {fold} = > train : {len(data["train"])} | val :{len(data["val"])}")

fold info : 
fold : 0 = > train : 500 | val :125
fold : 1 = > train : 500 | val :125
fold : 2 = > train : 500 | val :125
fold : 3 = > train : 500 | val :125
fold : 4 = > train : 500 | val :125


In [8]:
def run_fold(
        fold,
        full_train_images,
        full_valid_images,
        train_images,
        valid_images,
        valid_names,
        normalizer,
        train_transforms,
        test_transforms,
        loss_weights,
        ModelClass,
        LossFn,
        SchFn,
        HistoryRecorder,
        name_template,
        class_map,
        last_boot_level,
        losses_keys,
        args,

    ):


    ######################
    # set fce weights
    ######################

    class_counts = count_class_pixels(train_images=train_images,class_count=args["class_count"])
    alpha = effective_num_weights(class_counts,knee=20,beta=None)
    args["fce_conf"]["alpha"] = alpha.tolist()
    print("fce weigths : " ,args["fce_conf"]["alpha"])

    ######################
    # set fce weights make dataloaders
    ######################
    
    train_loader , valid_loader  = make_training_dataloaders(
        train_images=train_images,
        valid_images=valid_images,
        normalizer=normalizer,
        train_transforms=train_transforms,
        test_transforms=test_transforms,
        args=args,
        sampler=None
    )
    re_train_loader , re_valid_loader  = make_result_dataloaders(
        train_images=train_images,
        valid_images=full_valid_images,
        normalizer=normalizer,
        transforms=test_transforms
    )

    ######################
    # model init
    ######################

    model = ModelClass(args=args)
    model = model.to(args["device"])
    optimizer = torch.optim.AdamW(
        model.parameters(),
        **args["adamw_conf"]
    )
    loss_fn = LossFn(args)
    lr_sch = SchFn(optimizer=optimizer,total_iters=args["epochs"],power=0.9)
    recorder = HistoryRecorder(losses_keys=losses_keys,class_maps =class_map,
                            class_count=args["class_count"]-1)
    best_model = trainer(
        args=args,
        recorder = recorder,
        model = model,
        optimizer = optimizer,
        loss_fn = loss_fn,
        train_loader = train_loader,
        valid_loader = valid_loader,
        loss_weights=loss_weights,
        lr_sch = lr_sch,
        choice_score= args["socre_params"]["choice_score"],
    )

    ######################
    # model eval
    ######################
    
    args["name"] = name_template.format(
        best_model._get_name(),
        args["training_mode"],
        args["side_mode"],
        args["class_count"],
        args["epochs"],
        str(fold)
    )
    save_full_report(
        recorder= recorder , 
        output_base_path=args["paths"]["output_base_path"],
        model=best_model,
        valid_loader=valid_loader,
        normalizer = normalizer,
        args=args,
        class_map=class_map,
        name=args["name"],
        valid_images = valid_images,
        test_transforms = test_transforms,
        training_transforms = train_transforms,
        notebook_name = "Multi_Main.ipynb",
        class_count = args["class_count"],
        device = args["device"],
        training_mode = args["training_mode"],
        side_mode = args["side_mode"],
        last_boot_level=last_boot_level,
        fold_names = valid_names,

    )

    del best_model, model, recorder, lr_sch, loss_fn, optimizer
    del train_loader, valid_loader
    del re_train_loader, re_valid_loader
    return None

In [9]:
# name_template = args["name"]  
# for fold , data in tqdm(folds.items()):
#     print(f"=>>>>>>>>>>>>>>>>>>(FOLD {fold})<<<<<<<<<<<<<<<<<<<<=")

    
#     train_names = data["train"]
#     valid_names = data["val"]
#     train_images,valid_images = extract_fold_data(train_names,full_train_images)

#     run_fold(
#         fold=fold,
#         full_train_images=full_train_images,
#         full_valid_images=full_valid_images,
#         train_images=train_images,
#         valid_images=valid_images,
#         valid_names=valid_names,
#         normalizer=normalizer,
#         train_transforms=train_transforms,
#         test_transforms=test_transforms,
#         loss_weights=loss_weights,
#         ModelClass=SwinUperNet,
#         LossFn=UnetLoss,
#         SchFn=PolynomialLR,
#         HistoryRecorder=HistoryRecorder,
#         name_template=name_template,
#         class_map=class_map,
#         last_boot_level=last_boot_level,
#         losses_keys=losses_keys,
#         args=args,
#     )

#     gc.collect()
#     torch.cuda.empty_cache()

In [10]:
# folds_paths = {
#     0:["2026-08-01_12-15-26 [model[SwinUperNet]-binary-left-2class-30epo_fold0]",SwinUperNet],
#     1:["2026-08-01_12-38-11 [model[SwinUperNet]-binary-left-2class-30epo_fold1]",SwinUperNet],
#     2:["2026-08-01_13-00-57 [model[SwinUperNet]-binary-left-2class-30epo_fold2]",SwinUperNet],
#     3:["2026-08-01_13-23-43 [model[SwinUperNet]-binary-left-2class-30epo_fold3]",SwinUperNet],
#     4:["2026-08-01_13-46-28 [model[SwinUperNet]-binary-left-2class-30epo_fold4]",SwinUperNet],
# }

# folds_paths = {
#     0:["2026-08-01_21-19-26 [model[SwinUperNet]-binary-left-2class-30epo_fold0]",SwinUperNet],
#     1:["2026-08-01_21-42-09 [model[SwinUperNet]-binary-left-2class-30epo_fold1]",SwinUperNet],
#     2:["2026-08-01_22-04-52 [model[SwinUperNet]-binary-left-2class-30epo_fold2]",SwinUperNet],
#     3:["2026-08-01_22-27-36 [model[SwinUperNet]-binary-left-2class-30epo_fold3]",SwinUperNet],
#     4:["2026-08-01_22-50-20 [model[SwinUperNet]-binary-left-2class-30epo_fold4]",SwinUperNet],
# }
# folds_paths = {
#     0:["2026-08-08_14-07-22 [model[DSCNet_pro]-binary-left-2class-30epo_fold0]",DSCNet_pro],
#     1:["2026-08-08_14-43-08 [model[DSCNet_pro]-binary-left-2class-30epo_fold1]",DSCNet_pro],
#     2:["2026-08-08_15-18-55 [model[DSCNet_pro]-binary-left-2class-30epo_fold2]",DSCNet_pro],
#     3:["2026-08-08_15-54-38 [model[DSCNet_pro]-binary-left-2class-30epo_fold3]",DSCNet_pro],
#     4:["2026-08-08_16-30-31 [model[DSCNet_pro]-binary-left-2class-30epo_fold4]",DSCNet_pro],
# }
folds_paths = {
    0:["2026-08-14_14-57-19 [model[SwinUperNet]-multi-left-4class-50epo-Fold0]",SwinUperNet],
    1:["2026-08-14_15-35-06 [model[SwinUperNet]-multi-left-4class-50epo-Fold1]",SwinUperNet],
    2:["2026-08-14_16-13-03 [model[SwinUperNet]-multi-left-4class-50epo-Fold2]",SwinUperNet],
    3:["2026-08-14_16-50-46 [model[SwinUperNet]-multi-left-4class-50epo-Fold3]",SwinUperNet],
    4:["2026-08-14_17-28-29 [model[SwinUperNet]-multi-left-4class-50epo-Fold4]",SwinUperNet],
}
if(args["training_mode"] == "binary"):
    out_folder_path = folds_paths[0][0].split(" ")[0]
    out_base_path = Path(args["paths"]["base_path"])/"k_fold_preds"/ out_folder_path
    fracs = []
    th = 0.1
    os.makedirs(out_base_path / "result_vis" ,exist_ok=True)
    os.makedirs(out_base_path / "probs" ,exist_ok=True)
    os.makedirs(out_base_path / "extra_labels" ,exist_ok=True)
    with torch.no_grad():
        for fold , model_data in tqdm(folds_paths.items()):
            model_path , model_class= model_data
            out_folder = f"./outputs/{model_path}/"

            model,model_args,_ = load_model(
                model_class=model_class,
                path=out_folder,
                model_inputs={},
                device=args["device"],
                return_args=True
            )
            model.eval()

            input_image_base = Path(model_args["paths"]["base_path"]) / model_args["paths"]["images_folder"] / "train"
            input_mask_base = Path(model_args["paths"]["base_path"]) / model_args["paths"]["labels_folder"] / "train"

            with open(Path(out_folder) / "valid_names.json","r") as f:
                valid_names = json.load(f)
                valid_names = valid_names["val_names"]


            for name in valid_names:
                
                img = cv2.imread(input_image_base / f"{name}.png")
                mask = (np.load(input_mask_base/f"{name}.npy")!=0).astype(int)

                new_img= normalizer(
                    image = img,
                )["image"]
                new_img = torch.from_numpy(new_img).permute(2,0,1).unsqueeze(0).cuda()

                with torch.autocast("cuda"):
                    pred_logits = model(new_img)[0][0]
                    pred_probs = F.softmax(pred_logits,dim=1)[:,1]
                    pred_class = (pred_probs>th).int()

                pred_class = pred_class.cpu().numpy()[0]

                img = cv2.cvtColor(img,cv2.COLOR_RGB2GRAY)

                ### main logic
                # halo_band = cv2.dilate(mask.astype(np.uint8), np.ones((5,5), np.uint8)) & ~mask 
                # extra_preds = (pred_probs.cpu().numpy()[0] > th) & (mask == 0) & ~halo_band

                e_mask = cv2.dilate(mask.astype(np.uint8), np.ones((8,8), np.uint8))
                extra_preds = (pred_probs.cpu().numpy()[0]>th).astype(int)& (e_mask == 0)
                ####

                pred_class*=255
                mask*=255
                extra_preds *=255

                final_pl = draw_fast_plots([img,mask,pred_class,extra_preds],plot_shape=(2,2))

                cv2.imwrite(out_base_path / "result_vis"  /f"{name}.png",final_pl)

                np.save(out_base_path/ "probs" /f"{name}.npy",pred_probs.cpu().numpy()[0])
                np.save(out_base_path/ "extra_labels" /f"{name}.npy",(extra_preds//255).astype(int))
                fracs.append((extra_preds//255).astype(int).mean())

    fracs = np.array(fracs)
    print(f"mean {fracs.mean():.3%}  median {np.median(fracs):.3%}  max {fracs.max():.3%}")
    print("images ignoring >10%:", (fracs > 0.10).sum())
else:
    out_folder_path = folds_paths[0][0].split(" ")[0]
    out_base_path = Path(args["paths"]["base_path"])/"multi_k_fold_preds"/ out_folder_path
    fracs = []
    th = 0.1
    os.makedirs(out_base_path / "pred_probs" ,exist_ok=True)
    os.makedirs(out_base_path / "labels" ,exist_ok=True)
    with torch.no_grad():
        for fold , model_data in tqdm(folds_paths.items()):
            model_path , model_class= model_data
            out_folder = f"./outputs/{model_path}/"

            model,model_args,_ = load_model(
                model_class=model_class,
                path=out_folder,
                model_inputs={},
                device=args["device"],
                return_args=True
            )
            model.eval()

            input_image_base = Path(model_args["paths"]["base_path"]) / model_args["paths"]["images_folder"] / "train"
            input_mask_base = Path(model_args["paths"]["base_path"]) / model_args["paths"]["labels_folder"] / "train"

            with open(Path(out_folder) / "valid_names.json","r") as f:
                valid_names = json.load(f)
                valid_names = valid_names["val_names"]

            for name in tqdm(valid_names):
                # name = data["name_stem"]
                img = cv2.imread(input_image_base / f"{name}.png")
                mask = (np.load(input_mask_base/f"{name}.npy")!=0).astype(int)
                mask[mask!=0]-=4
                mask[mask==20]-=4
                mask[mask==21]-=4

                new_img= normalizer(
                    image = img,
                )["image"]
                new_img = torch.from_numpy(new_img).permute(2,0,1).unsqueeze(0).cuda()

                with torch.autocast("cuda"):
                    pred_logits = model(new_img)[0][0]
                    pred_probs = F.softmax(pred_logits,dim=1)
                    pred_class = torch.argmax(pred_probs,dim=1)

                pred_class = pred_class.cpu().numpy()[0]
                pred_probs = pred_probs.cpu().numpy()[0]
                
                np.save(out_base_path/ "pred_probs" /f"{name}.npy",pred_probs.astype(float))
                # np.save(out_base_path/ "pred_probs" /f"{fold}_{name}.npy",pred_probs.astype(float))

  0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/535 [00:00<?, ?it/s]

  0%|          | 0/125 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/535 [00:00<?, ?it/s]

  0%|          | 0/125 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/535 [00:00<?, ?it/s]

  0%|          | 0/125 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/535 [00:00<?, ?it/s]

  0%|          | 0/125 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/535 [00:00<?, ?it/s]

  0%|          | 0/125 [00:00<?, ?it/s]